# Financial Headline Sentiment Comparison

This notebook evaluates financial headline sentiment classification by comparing a specialized finance model (FinBERT) with a large language model (LLM) accessed through Groq.

The goal is to understand:

- How well a domain-specific transformer model (FinBERT) performs on financial news
- How its predictions compare with a general-purpose LLM
- Whether combining rule-based signals + FinBERT probabilities improves sentiment accuracy

## 1) Install and import libraries

Run the install cell once if your environment does not already have these packages.

In [ ]:
# pip install groq torch transformers scipy numpy pandas

In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd

from groq import Groq
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## 2) Create the dataset

These are the headlines we want to classify.

In [ ]:
headlines = [
    "Apple reports record quarterly earnings, beating analyst expectations",
    "Federal Reserve raises interest rates by 75 basis points",
    "Tesla stock plunges after disappointing delivery numbers",
    "Amazon announces massive layoffs amid economic slowdown",
    "Oil prices surge to highest level in three months",
    "Inflation data shows consumer prices rising faster than expected",
    "Microsoft acquires gaming giant in landmark $70 billion deal",
    "Unemployment rate falls to historic low of 3.4%",
    "Crypto market crashes as major exchange files for bankruptcy",
    "Goldman Sachs warns of recession risk in 2024",
    "Google parent Alphabet beats revenue forecasts on ad recovery",
    "Housing market cools as mortgage rates hit 20-year high",
    "Nvidia shares soar on AI chip demand boom",
    "Bank of America reports strong profit growth in Q3",
    "Supply chain disruptions ease, boosting manufacturing output",
    "Retail sales disappoint as consumers cut back on spending",
    "Meta posts surprise profit jump, stock rallies 20%",
    "China GDP growth slows sharply amid property crisis",
    "Warren Buffett increases stake in energy sector stocks",
    "Student loan repayments resume, threatening consumer spending",
    "JPMorgan raises dividend after passing Fed stress tests",
    "Pfizer cuts revenue forecast as COVID product demand fades",
    "US trade deficit narrows as exports climb",
    "WeWork files for bankruptcy protection",
    "Disney reports streaming losses narrowing, shares jump",
    "Elon Musk sells $4 billion in Tesla shares",
    "S&P 500 hits new all-time high on soft landing hopes",
    "Ford recalls 1.5 million vehicles over safety concerns",
    "Airbnb posts record revenue as travel demand stays strong",
    "Credit card delinquencies rise to highest level since 2010",
    "Berkshire Hathaway sits on record $157 billion cash pile",
    "Starbucks misses earnings estimates, lowers annual guidance",
    "US economy adds 300,000 jobs, crushing forecasts",
    "Moody's downgrades US credit outlook to negative",
    "Shopify revenue surges 25% on e-commerce rebound",
    "Boeing faces fresh safety scrutiny after door panel incident",
    "Fed signals potential rate cuts in 2024",
    "Oil giant ExxonMobil acquires Pioneer in $60 billion deal",
    "Snap shares collapse after weak advertising revenue",
    "IMF upgrades global growth forecast for 2024",
]

## 3) Build a simple rule-based sentiment score

This part gives each headline a lightweight sentiment score using keyword matching.

- bullish words push the score up
- bearish words push the score down
- the final rule-based score is between **-1 and 1**

This is not as smart as a language model, but it is fast and interpretable.

In [ ]:
BULLISH_WORDS = {
    "record", "beats", "beat", "surges", "surge", "soar", "soars", "strong",
    "growth", "profit", "rises", "rise", "high", "boom", "upgrades", "upgrade",
    "jump", "jumps", "rallies", "rally", "adds", "rebound", "narrows", "eases",
    "potential", "historic", "crushing",
}

BEARISH_WORDS = {
    "plunges", "plunge", "crashes", "crash", "layoffs", "recession", "cools",
    "disappoints", "disappoint", "slowdown", "crisis", "bankruptcy", "cuts",
    "cut", "falls", "fall", "delinquencies", "downgrades", "downgrade",
    "collapse", "collapses", "weak", "scrutiny", "sells", "concerns", "fades",
    "threatening", "slows",
}

def clean_words(text):
    return set(re.findall(r"\b[a-zA-Z']+\b", text.lower()))

def keyword_score(headline):
    words = clean_words(headline)
    bullish_hits = len(words & BULLISH_WORDS)
    bearish_hits = len(words & BEARISH_WORDS)

    total_hits = bullish_hits + bearish_hits
    if total_hits == 0:
        return 0.0

    return (bullish_hits - bearish_hits) / total_hits

## 4) Load FinBERT

We've discussed FinBERT previously, but as a refresher |
It's trained for sentiment tasks and
It returns probabilities for:
- positive
- negative
- neutral

We convert those probabilities into a single score using:

`positive probability - negative probability`

In [ ]:
MODEL_NAME = "ProsusAI/finbert" #loading in the model from hugging face
LABELS = ["positive", "negative", "neutral"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

In [ ]:
def finbert_predict(headline):
    inputs = tokenizer(headline, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = softmax(logits.numpy()[0])
    label = LABELS[int(np.argmax(probs))]
    score = float(probs[0] - probs[1])   # positive prob - negative prob

    return label, score, probs

## 5) Blend FinBERT with the keyword score

This creates one final sentiment score.

We give:
- **70% weight** to FinBERT
- **30% weight** to the keyword score (Not an official split, but this was an experiemt)

Then we turn the blended score into a label:
- above `0.10` → positive
- below `-0.10` → negative
- otherwise → neutral

In [ ]:
def blended_predict(headline, finbert_weight=0.7, threshold=0.10):
    finbert_label, finbert_score, probs = finbert_predict(headline)
    rule_score = keyword_score(headline)

    blended_score = finbert_weight * finbert_score + (1 - finbert_weight) * rule_score

    if blended_score > threshold:
        label = "positive"
    elif blended_score < -threshold:
        label = "negative"
    else:
        label = "neutral"

    return {
        "label": label,
        "blended_score": round(blended_score, 3),
        "finbert_label": finbert_label,
        "finbert_score": round(finbert_score, 3),
        "rule_score": round(rule_score, 3),
        "probs": probs,
    }

## 6) Connect to Groq

Now we send each headline to a Groq model and asks for **exactly one word**:

- positive
- negative
- neutral

For safety, the API key is loaded from an environment variable instead of being hard-coded in the notebook.

Before running the next cell, set your key:

```python
import os
os.environ["GROQ_API_KEY"] = "your_key_here"
```

Right now, I'm using a very simple prompt for Groq

In [1]:
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("Set GROQ_API_KEY before running the Groq cells.")

client = Groq(api_key=api_key)

NameError: name 'os' is not defined

In [ ]:
def groq_predict(headline, model_name="llama-3.1-8b-instant"):
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a financial sentiment classifier. "
                    "Reply with exactly one word: positive, negative, or neutral."
                ),
            },
            {"role": "user", "content": f"Headline: {headline}"},
        ],
        temperature=0,
        max_tokens=5,
    )

    return response.choices[0].message.content.strip().lower()

## 7) Run the full comparison

This loop:
- gets the blended sentiment
- gets the Groq sentiment
- checks whether they match
- stores everything in a table

In [ ]:
rows = []

for headline in headlines:
    blended = blended_predict(headline)
    groq_label = groq_predict(headline)
    match = blended["label"] == groq_label

    rows.append({
        "headline": headline,
        "combined_label": blended["label"],
        "blended_score": blended["blended_score"],
        "finbert_label": blended["finbert_label"],
        "finbert_score": blended["finbert_score"],
        "rule_score": blended["rule_score"],
        "groq_label": groq_label,
        "match": match,
    })

results_df = pd.DataFrame(rows)
results_df

## 8) Show a cleaner summary table

This view makes the output easier to read.

In [ ]:
display_df = results_df.copy()
display_df["match"] = display_df["match"].map({True: "Matched", False: "Not Matched"})
display_df[["headline", "combined_label", "blended_score", "groq_label", "match"]]

## 9) Compute agreement statistics

This tells us how often the blended system agrees with the Groq model.

In [ ]:
agreement_rate = results_df["match"].mean()

print(f"Agreement: {results_df['match'].sum()}/{len(results_df)} ({agreement_rate * 100:.1f}%)")

In [ ]:
summary_counts = pd.DataFrame({
    "combined": results_df["combined_label"].value_counts(),
    "groq": results_df["groq_label"].value_counts(),
}).fillna(0).astype(int)

summary_counts

## 10) Notes

You can extend this notebook by:
- testing different Groq models
- changing the FinBERT weight
- changing the positive/negative threshold
- adding more headlines
- plotting the score distribution